# 1.5 dalis (b) — Perkeltasis mokymas: tikslusis derinimas (angl. *fine-tuning*)
### `„Transfer learning 2“ Colab stiliuje — užšaldytos bazės laipsniškas atšaldymas`

**Kaip veikia originalus „Colab“ (Transfer learning 2):**
1. Pirmiausia mokoma **tik galvutė** su užšaldyta baze (žr. 1.5(a));
2. Tada bazės viršutiniai sluoksniai **atšaldomi**, o likusieji lieka užšaldyti;
3. Modelis mokomas toliau su **labai mažu mokymosi greičiu** (pvz., `1e-5`),
   kad atšaldyti svoriai nebūtų sugadinti;
4. Rezultatas: bazinis modelis prisitaiko prie naujų duomenų specifikos — paprastai
   pasiekiamas dar keliais procentais didesnis tikslumas.

Šioje užrašinėje abu etapai atliekami nuosekliai (savarankiškas failas):
**1 etapas** — požymių išskyrimas (angl. *feature extraction*), **2 etapas** — tikslusis derinimas (angl. *fine-tuning*) ir palyginimas.

> Rekomenduojama naudoti GPU: *Runtime → Change runtime type →* **T4 GPU**.


## 1. Aplinka ir duomenys (kaip ir 1.5(a))

In [ ]:
%pip install -q tensorflow Pillow scikit-learn pandas matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

import tensorflow as tf
from PIL import Image, ImageDraw, ImageFont
import textwrap
tf.random.set_seed(SEED)
print('TensorFlow version:', tf.__version__)

In [ ]:
# --- Deterministic dataset generator (same as scripts/generate_dataset.py) ---
# X = customer query text, y = intent label (6 classes).
DATA_SPEC = {
    'order_status': [
        ('Kur mano užsakymas {order}?', 'order'),
        ('Kokia yra užsakymo {order} būsena?', 'order'),
        ('Nematau užsakymo {order} paskyroje, ką daryti?', 'order'),
        ('Kaip sužinoti mano užsakymo {order} statusą?', 'order'),
        ('Ar užsakymas {order} jau išsiųstas?', 'order'),
        ('Mano užsakymo {order} statusas vis dar „Apmokėta“, ar tai normalu?', 'order'),
        ('Kada bus matoma užsakymo {order} sekimo informacija?', 'order'),
    ],
    'delivery': [
        ('Kiek trunka pristatymas į {city}?', 'city'),
        ('Ar pristatote siuntas į {city} savaitgaliais?', 'city'),
        ('Kokia pristatymo į {city} kaina?', 'city'),
        ('Kada mano siunta pasieks {city}?', 'city'),
        ('Ar taikomas nemokamas pristatymas perkant už {eur} eurų?', 'eur'),
        ('Kur rasti siuntos į {city} sekimo numerį?', 'city'),
        ('Ar galiu atsiimti siuntą į {city} paštomate?', 'city'),
    ],
    'returns': [
        ('Noriu grąžinti {item}, kaip tai padaryti?', 'item'),
        ('Ar galiu grąžinti prekę po {days} dienų?', 'days'),
        ('Kiek kainuoja grąžinti {item}?', 'item'),
        ('Kada atgausiu pinigus grąžinęs {item}?', 'item'),
        ('Ar galima grąžinti {item} originalioje pakuotėje?', 'item'),
        ('Kur rasiu grąžinimo formą norėdamas grąžinti {item}?', 'item'),
        ('Ar galima grąžinti {item}, jei pakuotė jau buvo atidaryta?', 'item'),
    ],
    'payment': [
        ('Ar priimate {method}?', 'method'),
        ('Kiek laiko trunka mokėjimo autorizacija pasirinkus {method}?', 'method'),
        ('Nepavyko apmokėti užsakymo pasirinkus {method}, ką daryti?', 'method'),
        ('Ar galima mokėti dalimis pasirinkus {method}?', 'method'),
        ('Kada gausiu sąskaitą faktūrą apmokėjus per {method}?', 'method'),
        ('Ar palaikote {method} internetinėje parduotuvėje?', 'method'),
        ('Ką daryti, jei mokant per {method} nurašė pinigus, bet užsakymas neapmokėtas?', 'method'),
    ],
    'warranty': [
        ('Kiek laiko galioja garantija įsigijus {item}?', 'item'),
        ('Ar nemokamai suremontuosite {item} pagal garantiją?', 'item'),
        ('Ką daryti, jei gaminys sugedo po {days} dienų naudojimo?', 'days'),
        ('Ar reikia atskiro talono pateikiant {item} garantiniam remontui?', 'item'),
        ('Kiek trunka garantinis remontas pateikus {item}?', 'item'),
        ('Ar garantija galioja, jei netyčia pažeidžiau {item}?', 'item'),
        ('Ar atvežate {item} į remontą nemokamai?', 'item'),
    ],
    'product_usage': [
        ('Kaip susieti {item} su telefonu?', 'item'),
        ('Kur rasti naudotojo vadovą įsigijus {item}?', 'item'),
        ('Nusipirkau {item}, bet prietaisas neįsijungia, ką daryti?', 'item'),
        ('Kaip atstatyti gamyklinius nustatymus naudojant {item}?', 'item'),
        ('Ar galima gauti naudotojo vadovą lietuvių kalba įsigijus {item}?', 'item'),
        ('Kaip tinkamai prijungti {item} prie kompiuterio?', 'item'),
        ('Kiek laiko reikia krauti {item}?', 'item'),
    ],
}

SLOT_VALUES = {
    'order': ['TS-102345', 'TS-201111', 'TS-314560', 'TS-456789', 'TS-511235', 'TS-623456'],
    'city': ['Vilnių', 'Kauną', 'Klaipėdą', 'Šiaulius', 'Panevėžį', 'Alytų'],
    'item': ['išmanųjį telefoną', 'belaides ausines', 'išorinę bateriją',
             'elektroninę skaityklę', 'žaidimų pultelį', 'internetinę kamerą'],
    'days': ['3', '7', '10', '13', '20', '30'],
    'eur': ['30', '45', '60', '75', '90', '120'],
    'method': ['PayPal', 'Visa kortelę', 'Mastercard kortelę',
               'bankinį pavedimą', 'Google Pay', 'Apple Pay'],
}

INTENTS = ['order_status', 'delivery', 'returns', 'payment', 'warranty', 'product_usage']


def generate_dataset():
    rows = []
    for intent, specs in DATA_SPEC.items():
        for template, slot in specs:
            for value in SLOT_VALUES[slot]:
                rows.append((template.replace('{' + slot + '}', value), intent))
    return pd.DataFrame(rows, columns=['query', 'intent'])


df = generate_dataset()
print(f'Dataset: {len(df)} (X, y) pairs, {len(INTENTS)} classes')
print(df['intent'].value_counts())

# Canonical hold-out test set (same 6 queries as data/customer_support_test.csv
# and as the 1.2.2 Gemini prototype) - used to compare all models fairly.
TEST_EXAMPLES = [
    ('Kokiomis dienomis pristatote užsakymus į Vilnių?', 'delivery'),
    ('Norėčiau grąžinti prekę po 10 dienų nuo pirkimo, ar įmanoma?', 'returns'),
    ('Nematau savo užsakymo istorijos paskyroje, ką daryti?', 'order_status'),
    ('Ar parduotuvė priima PayPal mokėjimus?', 'payment'),
    ('Ar už 25 eurus pirktas gaminys turi garantiją?', 'warranty'),
    ('Kur rasti ausinių susiejimo instrukciją?', 'product_usage'),
]

IMG_SIZE = 96

def get_font(size=16):
    for path in ['/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
                 'C:/Windows/Fonts/arial.ttf']:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            continue
    return ImageFont.load_default()

def render_query_rgb(text, idx, img_size=IMG_SIZE):
    rng = np.random.RandomState(idx)
    img = Image.new('RGB', (img_size, img_size), color=(235, 235, 235))
    draw = ImageDraw.Draw(img)
    wrapped = textwrap.wrap(text, width=13)[:4]
    y = 8 + rng.randint(-2, 3)
    for line in wrapped:
        draw.text((6 + rng.randint(-2, 3), y), line, fill=(20, 20, 20), font=get_font(15))
        y += 18
    return np.asarray(img, dtype='float32')

X_images = np.stack([render_query_rgb(q, i) for i, q in enumerate(df['query'])])
y_labels = np.array([INTENTS.index(l) for l in df['intent']])
print('RGB image dataset:', X_images.shape)

## 2. 1 etapas — galvutės mokymas su užšaldyta baze
(Tas pats kaip ir 1.5(a); trumpesnis mokymas — čia tai tik pradinis atskaitos taškas.)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train_raw, y_val_raw = train_test_split(
    X_images, df['intent'], test_size=0.2, random_state=SEED, stratify=df['intent'])
y_train = np.array([INTENTS.index(l) for l in y_train_raw])
y_val = np.array([INTENTS.index(l) for l in y_val_raw])
X_test = np.stack([render_query_rgb(q, 30_000 + i) for i, (q, _) in enumerate(TEST_EXAMPLES)])
y_test = np.array([INTENTS.index(l) for _, l in TEST_EXAMPLES])

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = tf.keras.layers.Rescaling(1.0 / 127.5, offset=-1.0)(inputs)  # MobileNetV2 expects [-1, 1]
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(INTENTS), activation='softmax')(x)
tl_model = tf.keras.Model(inputs, outputs)

tl_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                 loss='sparse_categorical_crossentropy', metrics=['accuracy'])

stage1_history = tl_model.fit(
    X_train, y_train, validation_data=(X_val, y_val),
    epochs=5, batch_size=32, verbose=2)

stage1_val_acc = max(stage1_history.history['val_accuracy'])
print(f'1 etapas (požymių išskyrimas) — validacijos tikslumas: {stage1_val_acc:.4f}')

## 3. 2 etapas — tikslusis derinimas (angl. *fine-tuning*): atšaldome viršutinius sluoksnius

`MobileNetV2` turi 154 sluoksnius. Atšaldome **paskutinius 30**, o
`BatchNormalization` sluoksniai lieka užšaldyti — jų statistika paimta iš
„ImageNet“ duomenų. Mokymosi greitis sumažinamas 100 kartų (`1e-3 → 1e-5`),
kitaip dideli gradientai sugadintų perkeltus svorius.

In [ ]:
base_model.trainable = True
# keep BatchNormalization layers frozen - their statistics come from ImageNet
for layer in base_model.layers[:-30]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

tl_model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                 loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainable = [v.name for v in tl_model.trainable_variables]
print(f'Trainable tensors after unfreezing: {len(trainable)}')
# must recompile -> metrics reset, fine to fit from current weights
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=5, restore_best_weights=True)

initial_epoch = len(stage1_history.history['loss'])
stage2_history = tl_model.fit(
    X_train, y_train, validation_data=(X_val, y_val),
    initial_epoch=initial_epoch,
    epochs=initial_epoch + 10, batch_size=32,
    callbacks=[early_stop], verbose=2)

stage2_val_acc = max(stage2_history.history['val_accuracy'])
print(f'2 etapas (tikslusis derinimas) — validacijos tikslumas: {stage2_val_acc:.4f}')

## 4. Abiejų etapų kreivės ir palyginimas

In [ ]:
acc = stage1_history.history['accuracy'] + stage2_history.history['accuracy']
val_acc = stage1_history.history['val_accuracy'] + stage2_history.history['val_accuracy']
loss = stage1_history.history['loss'] + stage2_history.history['loss']
val_loss = stage1_history.history['val_loss'] + stage2_history.history['val_loss']
boundary = len(stage1_history.history['loss'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(loss, label='mokymo nuostoliai')
axes[0].plot(val_loss, label='validacijos nuostoliai')
axes[0].axvline(boundary - 0.5, color='gray', linestyle='--', label='tikslusis derinimas prasideda')
axes[0].set_xlabel('Epocha'); axes[0].set_ylabel('Nuostoliai (Loss)'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(acc, label='mokymo tikslumas')
axes[1].plot(val_acc, label='validacijos tikslumas')
axes[1].axvline(boundary - 0.5, color='gray', linestyle='--', label='tikslusis derinimas prasideda')
axes[1].set_xlabel('Epocha'); axes[1].set_ylabel('Tikslumas (Accuracy)'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle('1 etapas (požymių išskyrimas) ir 2 etapas (tikslusis derinimas)')
plt.tight_layout(); plt.show()

In [ ]:
def plot_confusion_matrix(y_true, y_pred, classes, title):
    cm = np.zeros((len(classes), len(classes)), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    fig, ax = plt.subplots(figsize=(5.5, 5))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(len(classes)), classes, rotation=45, ha='right')
    ax.set_yticks(range(len(classes)), classes)
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else 'black')
    ax.set_xlabel('Numatytas ketinimas')
    ax.set_ylabel('Tikrasis ketinimas')
    ax.set_title(title)
    fig.colorbar(im)
    plt.tight_layout()
    plt.show()
    return cm

from sklearn.metrics import accuracy_score

val_pred = np.argmax(tl_model.predict(X_val, verbose=0), axis=1)
test_pred = np.argmax(tl_model.predict(X_test, verbose=0), axis=1)
test_acc = accuracy_score(y_test, test_pred)

print(f'Galutinis validacijos tikslumas: {accuracy_score(y_val, val_pred):.4f}')
print(f'Galutinis testavimo (6 užklausos) tikslumas: {test_acc:.4f}')
print()
print('Modelių palyginimas (tas pats uždavinys):')
print(f'  1 etapas (požymių išskyrimas): val={stage1_val_acc:.4f}')
print(f'  2 etapas (tikslusis derinimas): val={stage2_val_acc:.4f}  test={test_acc:.4f}')
print()
for (query, true_label), pred_id in zip(TEST_EXAMPLES, test_pred):
    mark = 'OK ' if INTENTS[pred_id] == true_label else 'WRONG'
    print(f'{mark} true={true_label:14s} pred={INTENTS[pred_id]:14s} {query}')

plot_confusion_matrix(y_val, val_pred, INTENTS,
                      'Tiksliai suderintas modelis — painiavos matrica')

## 5. Išvados
1. Paskutinių sluoksnių tikslusis derinimas (angl. *fine-tuning*) su 100× mažesniu mokymosi greičiu yra
   saugus būdas pritaikyti iš anksto apmokytą modelį savo duomenims.
2. Jei duomenų yra nedaug ir jie labai skiriasi nuo „ImageNet“ (mūsų atveju —
   teksto piešiniai), tiksliojo derinimo nauda būna nedidelė; su natūraliomis
   nuotraukomis ji dažniausiai yra reikšmingesnė.
3. `BatchNormalization` sluoksnių užšaldymas — svarbi praktinė detalė: kitaip
   jų išmokta statistika būtų iškreipta dėl mažų paketų (angl. *batch*) imčių.
4. Pilnas projekto modelių palyginimas: matyti, kad tekstinėms
   klientų užklausoms geriausiai tinka TF-IDF + (MLP arba sprendimų medžiai), o vaizdų metodai
   (CNN / perkeltasis mokymas) iliustruoja principą ir būtų tinkamesni
   dirbant su realiais vaizdų duomenimis.
